# mT5 vs. NLLB: architecture comparison on English/Kiswahili <-> Ekegusii

**Purpose:** a direct comparison against `02_nllb_training.ipynb` on the same task
(English<->Ekegusii, Kiswahili<->Ekegusii), evaluated on `test_psa` alone - one overall
number per direction, no bible/general split, no domain breakdown - so this notebook's
`psa` result sits directly against `02`'s own `psa` column. Not the 4-way ablation
`mt5_training.ipynb` ran before either (stage1,
stage2, mixed, psa_only) - just **one** standalone run: `mixed` curriculum (the full
dataset, not PSA-only), `use_lora=False` (layer-freeze, not LoRA) - fixed choices,
matching `BEST_NLLB_CHECKPOINT.json`'s recorded winner (`mixed_lora_only`) directly, not
a dynamic branch computed at runtime. Trained once, directly from `google/mt5-small` -
never chained from another experiment's checkpoint. `psa_only` and standalone `stage1`
are out of scope on purpose - the question is whether the winning *curriculum and
training method* transfers to a second architecture, not "which of four mT5 configs is
best."

**No blocking dependency**: unlike an earlier version of this notebook, nothing here is
computed from `02_nllb_training.ipynb`'s ablation at runtime - `mixed`/`use_lora=False`
are fixed. Section 5 only cross-checks that fixed choice against
`BEST_NLLB_CHECKPOINT.json` if that file happens to exist, and warns (doesn't stop) if it
ever disagrees - this notebook can run start to finish with or without it, and none of it
needs the GPU until Section 6 (training).

## 1. Setup

In [1]:
!pip install -q torch transformers accelerate sentencepiece evaluate sacrebleu mlflow peft matplotlib pandas datasets


### GPU / CUDA check — run this before anything else

In [2]:
import torch
print(f"torch version       : {torch.__version__}")
print(f"CUDA available      : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU                 : {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: no GPU visible to torch - fix this before continuing.")


torch version       : 2.11.0+cu128
CUDA available      : True
GPU                 : NVIDIA H100 PCIe


In [3]:
import sys, os
from pathlib import Path

def find_project_root(marker="arch_config.py"):
    current = Path.cwd().resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not find {marker} by walking up from {current}")

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f"project root: {PROJECT_ROOT}")
if "final_training" not in str(PROJECT_ROOT).lower().replace(" ", "_"):
    print("\n  WARNING: 'Final_Training' doesn't appear in PROJECT_ROOT above - this notebook "
          "must live under Final_Training/Notebooks/ for find_project_root() to land in the "
          "right place. Move it there before continuing.")

from arch_config import ARCH_CONFIG, MAX_LEN, LORA_KWARGS, SEED, DATA_DIR, CHECKPOINTS_DIR, LOGS_DIR, DEPLOYMENT_DIR
import data_io, metrics, inference
from train import train_stage
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, set_seed
from datasets import Dataset
import pandas as pd

set_seed(SEED)
ARCHITECTURE = "mt5"
cfg = ARCH_CONFIG[ARCHITECTURE]
print(f"architecture: {ARCHITECTURE}")
print(f"checkpoint  : {cfg['checkpoint']}")


project root: /home/jovyan/PSA/Final_Training
architecture: mt5
checkpoint  : google/mt5-small


## 2. Load training data (`mixed`) and the PSA test set

One test set, not three - `test_psa` only, evaluated as a whole (no bible/general split,
no domain breakdown). `mixed`/`validation_psa` for training.

In [4]:
curriculum = data_io.load_curriculum(DATA_DIR)
test_sets = {"test_psa": data_io.load_jsonl(Path(DATA_DIR) / "test_psa.jsonl")}
print(f"test_psa: {len(test_sets['test_psa']):,} examples")

for name in ["mixed", "validation_psa"]:
    print(f"  {name:20s} {len(curriculum[name]):>8,} rows")


test_psa: 3,020 examples
  mixed                 227,802 rows
  validation_psa          3,405 rows


## 3. Tokenizer and tokenization

`mixed`/`validation_psa` for training, `test_psa` for evaluation.

In [5]:
tokenizer = AutoTokenizer.from_pretrained(cfg["checkpoint"])

def mt5_preprocess(batch):
    inputs = [inference.mt5_prefix(sl) + s for sl, s in zip(batch["src_lang"], batch["src"])]
    model_inputs = tokenizer(inputs, max_length=MAX_LEN, truncation=True)
    labels = tokenizer(text_target=batch["tgt"], max_length=MAX_LEN, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

def tokenize_rows(rows):
    ds = Dataset.from_list(rows)
    ds = ds.map(lambda r: {"src_lang": "en" if r["src_lang"] == "eng_Latn" else "sw"})
    return ds.map(mt5_preprocess, batched=True, batch_size=1000)

tok = {name: tokenize_rows(curriculum[name]) for name in ["mixed", "validation_psa"]}
tok["test_psa"] = tokenize_rows(test_sets["test_psa"])
print("tokenization complete -", list(tok.keys()))


Map:   0%|          | 0/227802 [00:00<?, ? examples/s]

Map:   0%|          | 0/227802 [00:00<?, ? examples/s]

Map:   0%|          | 0/3405 [00:00<?, ? examples/s]

Map:   0%|          | 0/3405 [00:00<?, ? examples/s]

Map:   0%|          | 0/3020 [00:00<?, ? examples/s]

Map:   0%|          | 0/3020 [00:00<?, ? examples/s]

tokenization complete - ['mixed', 'validation_psa', 'test_psa']


## 4. Zero-shot baseline

No training, just the base checkpoint saved for evaluation - also unconditional, no GPU
needed for this (no `.to("cuda")`, no generation, just a load-then-save), safe to run
alongside `02_nllb_training.ipynb`'s currently-running GPU job.

In [6]:
zeroshot_path = f"{CHECKPOINTS_DIR}/mt5_zeroshot"
AutoModelForSeq2SeqLM.from_pretrained(cfg["checkpoint"]).save_pretrained(zeroshot_path, safe_serialization=True)
tokenizer.save_pretrained(zeroshot_path)
print(f"zero-shot baseline saved: {zeroshot_path}")


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

zero-shot baseline saved: checkpoints/mt5_zeroshot


## 5. Confirm this matches the recorded NLLB winner

Fixed, not derived: this notebook trains **only** `mixed`, **only** `use_lora=False` -
that's the whole point of "standalone," not a branch that could go either way depending
on what some other file says. This cell just cross-checks that fixed choice against
`BEST_NLLB_CHECKPOINT.json` (the same record `03_nllb_other_languages.ipynb` reads) and
prints a loud warning - it does **not** change what gets trained - if that record ever
names something other than `mixed_lora_only` as the winner, so a future reader isn't
misled about why "mixed, no LoRA" was the choice.

In [7]:
import json

CURRICULUM = "mixed"
USE_LORA = False

best_checkpoint_record = PROJECT_ROOT / "BEST_NLLB_CHECKPOINT.json"
if best_checkpoint_record.exists():
    record = json.loads(best_checkpoint_record.read_text())
    recorded_run = record["run_id"]
    expected_run = f"{CURRICULUM}_lora_only" if not USE_LORA else f"{CURRICULUM}_lora"
    if recorded_run != expected_run:
        print(f"WARNING: BEST_NLLB_CHECKPOINT.json now names {recorded_run!r} as the NLLB "
              f"winner, not {expected_run!r} - this notebook still trains {CURRICULUM!r} / "
              f"use_lora={USE_LORA} regardless (that choice is fixed, not re-derived), but "
              f"the record this was supposed to mirror has apparently changed.")
    else:
        print(f"confirmed: matches the recorded NLLB winner ({recorded_run!r}).")
else:
    print(f"{best_checkpoint_record.name} not found - proceeding anyway with the fixed "
          f"choice ({CURRICULUM!r}, use_lora={USE_LORA}); this file is only used here as "
          f"a cross-check, not a dependency.")


BEST_NLLB_CHECKPOINT.json not found - proceeding anyway with the fixed choice ('mixed', use_lora=False); this file is only used here as a cross-check, not a dependency.


## 6. Train — one standalone run, `mixed` only, no LoRA

A single training call, directly from `google/mt5-small` (`cfg["checkpoint"]`) - not
chained from another experiment's output, not branching on anything computed elsewhere.

In [8]:
import logging
import warnings

logging.getLogger("mlflow").setLevel(logging.CRITICAL)
warnings.filterwarnings("ignore")

In [9]:
final_run_name = "mt5_mixed"
_, final_path, _ = train_stage(
    final_run_name, ARCHITECTURE, cfg, "mixed", cfg["checkpoint"],
    tok["mixed"], tok["validation_psa"], tokenizer, MAX_LEN, LORA_KWARGS,
    use_lora=USE_LORA, checkpoints_dir=CHECKPOINTS_DIR, logs_dir=LOGS_DIR)

print(f"final mT5 checkpoint (mixed, use_lora={USE_LORA}): {final_path}")


  mt5_mixed   arch=mt5  stage=mixed  init=google/mt5-small  use_lora=False
  checkpoint -> /home/jovyan/PSA/Final_Training/checkpoints/mt5_mixed
  mlflow db  -> /home/jovyan/PSA/Final_Training/mlflow.db


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Epoch,Training Loss,Validation Loss,Bleu,Chrf2pp
1,2.432211,1.948981,11.013040,32.214981
2,2.074890,1.663402,13.935508,36.335591
3,1.816761,1.542423,16.063222,39.460488
4,1.783251,1.461490,18.542363,41.089521
5,1.668059,1.415800,18.892084,41.748245
6,1.617840,1.385219,19.943210,42.622556
7,1.563116,1.370485,20.651017,42.930213
8,1.540321,1.366476,20.522721,43.123763


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


mt5_mixed training time: 131.1 minutes


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

final mT5 checkpoint (mixed, use_lora=False): checkpoints/mt5_mixed/merged


## 7. Evaluate: zero-shot vs. fine-tuned, `test_psa` only, both directions

One test set, evaluated as a whole - overall chrF2++/BLEU per run per direction, no
bible/general split, no domain breakdown. `02_nllb_training.ipynb`'s own `psa` column is
the direct comparison point (same underlying `test_psa` data, same metric).

In [10]:
test_en = [r for r in test_sets["test_psa"] if r["src_lang"] == "eng_Latn"]
test_sw = [r for r in test_sets["test_psa"] if r["src_lang"] == "swh_Latn"]
print(f"test_psa sizes: {len(test_en):,} English-source, {len(test_sw):,} Kiswahili-source")

all_paths = {"zeroshot": zeroshot_path, final_run_name.replace("mt5_", ""): str(final_path)}
results = []
for run_name, path in all_paths.items():
    model = AutoModelForSeq2SeqLM.from_pretrained(path)
    model.to("cuda" if torch.cuda.is_available() else "cpu")
    tok_for_run = AutoTokenizer.from_pretrained(path)

    # generate_one() in a loop, not a batched helper - inference.py doesn't have one, and
    # test_psa is small (2,000 rows), so there's no real performance need for one either.
    preds_en = [inference.generate_one(model, tok_for_run, r["src"], "en", cfg, ARCHITECTURE, MAX_LEN)
                for r in test_en]
    results.append(metrics.score(preds_en, [r["tgt"] for r in test_en], f"mt5_{run_name}_en-guz"))

    preds_sw = [inference.generate_one(model, tok_for_run, r["src"], "sw", cfg, ARCHITECTURE, MAX_LEN)
                for r in test_sw]
    results.append(metrics.score(preds_sw, [r["tgt"] for r in test_sw], f"mt5_{run_name}_sw-guz"))

    del model
    torch.cuda.empty_cache()

results_df = pd.DataFrame(results)[["name", "chrf2pp", "bleu"]]
results_df.to_csv(f"{LOGS_DIR}/mt5_psa_results.csv", index=False)
print(results_df.to_string(index=False))


test_psa sizes: 1,862 English-source, 1,158 Kiswahili-source


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


mt5_zeroshot_en-guz                 chrF2++=1.02  BLEU=0.00
mt5_zeroshot_sw-guz                 chrF2++=1.39  BLEU=0.01


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


mt5_mixed_en-guz                    chrF2++=29.37  BLEU=5.60
mt5_mixed_sw-guz                    chrF2++=27.62  BLEU=3.97
               name   chrf2pp     bleu
mt5_zeroshot_en-guz  1.021260 0.002304
mt5_zeroshot_sw-guz  1.391398 0.013710
   mt5_mixed_en-guz 29.366176 5.595365
   mt5_mixed_sw-guz 27.622917 3.966457


## 8. Save deployment-ready checkpoint

In [11]:
for run_name, path in all_paths.items():
    inference.save_for_deployment(path, ARCHITECTURE, f"mt5_{run_name}", deployment_dir=DEPLOYMENT_DIR)


deployment-ready: /home/jovyan/PSA/Final_Training/deployment/mt5/mt5_zeroshot  (1 safetensors file(s))
deployment-ready: /home/jovyan/PSA/Final_Training/deployment/mt5/mt5_mixed  (1 safetensors file(s))


## 9. Demonstration

PSA-register samples only, matching the PSA-only evaluation scope above.

In [12]:
demo_model = AutoModelForSeq2SeqLM.from_pretrained(final_path)
demo_model.to("cuda" if torch.cuda.is_available() else "cpu")
demo_tokenizer = AutoTokenizer.from_pretrained(final_path)

samples = [
    ("Farmers are urged to prioritize safe agrochemical usage this season.", "en"),   # PSA register
    ("Ministry of Health: report suspected cholera cases immediately.", "en"),         # PSA register
    ("Wakulima wanahimizwa kutumia kemikali za kilimo kwa usalama msimu huu.", "sw"),  # PSA register
]

print(f"mT5 demo - configuration: mixed, use_lora={USE_LORA}\n")
for text, lang in samples:
    print(f"[{lang}] {text}")
    print("  ->", inference.generate_one(demo_model, demo_tokenizer, text, lang, cfg, ARCHITECTURE, MAX_LEN))
    print()


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


mT5 demo - configuration: mixed, use_lora=False

[en] Farmers are urged to prioritize safe agrochemical usage this season.
  -> Abakoriria abakubuokho kutegeka kwa kumwita amaroba yasiyotokorwa kwa mwaka wa mwaka

[en] Ministry of Health: report suspected cholera cases immediately.
  -> Eminisitiri yechisemi: Ekeombe gia abanyagitari nigo kireteire amang'ana y'oborwari bwa cholera ase ogosima.

[sw] Wakulima wanahimizwa kutumia kemikali za kilimo kwa usalama msimu huu.
  -> Abakoriria abakubuokho kusoma matundu ya kilimo kwa mwaka huu

